In [ ]:
import earthkit.hydro as ekh
import numpy as np
import xarray as xr

# Handling xarray and array inputs and outputs

For every operation, earthkit-hydro has two versions:
- an xarray version, called at `ekh.submodule.operation`
- an array version, called at `ekh.submodule.array.operation`

This notebook demonstrates this for some simple operations.

In [ ]:
network = ekh.river_network.load("efas", "5")

## xarray versions

xarray is the high-level recommended approach for conducting operations on river networks.

For this tutorial, example xarray data is created, but in practice this can be loaded from any supported file with
`ekd.from_source(...).to_xarray()` (or `xr.open_dataset(...)`).

In [ ]:
example_arr = np.random.rand(2, *network.shape)

lat = network.coords['lat']
lon = network.coords['lon']
step = np.arange(2)

example_da = xr.DataArray(
    example_arr,
    dims = ["step", "lat", "lon"],
    coords = {"step": step, "lat": lat, "lon": lon},
    name = "precip",
    attrs={"units": "m", "description": "Sample precipitation data"}
)
example_da

All operations can then be directly conducted on this xarray dataarray, and will return back an xarray dataarray.

For example, we can do an `upstream.sum`, or a `catchments.sum`.

In [ ]:
ekh.upstream.sum(network, example_da)

In [ ]:
ekh.catchments.sum(network, example_da, locations = {"station_1": (60,60), "station_2": (50,-120)} )

We can also pass a dataset, in which case we will be returned back a corresponding dataset.

In [ ]:
example_ds = xr.Dataset(
    data_vars={"var1": example_da, "var2": example_da+1}
)
example_ds

In [ ]:
ekh.upstream.sum(network, example_ds)

Note that the coordinates were automatically detected in the above example. In some cases, it will not be possible infer the spatial coordinates in the xarray object. See for example the following:

In [ ]:
example_da_uninferrable = xr.DataArray(
    example_arr,
    dims = ["step", "uninferrable_name_1", "uninferrable_name_2"],
    coords = {"step": step, "uninferrable_name_1": lat, "uninferrable_name_2": lon},
    name = "precip",
    attrs={"units": "m", "description": "Sample precipitation data"}
)

This will raise a `ValueError: Could not autodetect xarray core dims`. However, we can manually specify the input core dimensions for each of the xarray inputs via the argument `input_core_dims`.

In [ ]:
ekh.upstream.sum(network, example_da_uninferrable, input_core_dims=[["uninferrable_name_1", "uninferrable_name_2"]])

We need to specify this for each xarray argument. In the below example, we have two xarray arguments so we specify the input_core_dims for both, in the order defined by the function signature.

In [ ]:
ekh.upstream.sum(network, example_da, node_weights=example_da_uninferrable, input_core_dims=[["lat", "lon"], ["uninferrable_name_1", "uninferrable_name_2"]])

It is also possible to give purely array inputs, in which case an xarray dataarray is returned with the coordinate information of the river network. For example,

In [ ]:
ekh.upstream.sum(network, example_arr)

**Note:** Array inputs must follow the convention that the *last* dimensions are the river network related dimensions, in this case lat and lon. Therefore, for any vectorised operations, the extra dimensions must be *leading*.

## array versions

A lower level API is also available for users interested in working directly with arrays. This API does not accept any xarray inputs, but works analagously to the xarray API. Taking the two above examples, the array forms are:

In [ ]:
ekh.upstream.array.sum(network, example_arr)

In [ ]:
ekh.catchments.array.sum(network, example_arr, locations = {"station_1": (60,60), "station_2": (50,-120)})